# 03 · Produccion multifuente (N bases) — nivel AVANZADO

**Que hace**: consolida **2, 3, 4 o mas fuentes** en un registro unico empresarial:
deduplica dentro de cada fuente y cruza entre fuentes en una sola corrida
(`linkage`, el mismo Orchestrator del flujo de produccion), con fuentes confiables
(`trusted`), refinador multicampo opcional y **manifiesto de trazabilidad** en JSON.

**Disenado para** Google Colab Free: ZIP/CSV/TXT/XLSX se leen con contratos
`SourceSpec`; descompresion y checkpoints viven en el disco local `/content`.

**Pasos**: Celda A (entorno) → Celda A2 (definiciones) → **Celda B** (config + correr).

> Motor: `rues-linker 0.22.4`. Salidas: GOLDEN, CORRELATIVA, MATRIZ_FUENTES,
> RESUMEN + `manifiesto_corrida.json`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.22.4"
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"


def _version_ok(v: str, objetivo: str = VERSION_OBJETIVO) -> bool:
    """True si la version instalada coincide con el release objetivo."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) == _t(objetivo)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza exactamente rues-linker 0.22.4 desde el tag oficial."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada):
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub...")
    else:
        print(f"rues-linker {instalada} no coincide con {VERSION_OBJETIVO}; instalando...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada. "
            "Reinicie el entorno y vuelva a ejecutar esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_OBJETIVO}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════

# ── Lectura y validacion de bases (fail-fast con mensajes accionables) ──────

import gc
import hashlib
import json
import time
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from record_linkage import (
    ColumnType,
    LoadedSource,
    SourceSpec,
    ZipSafetyLimits,
    __version__ as _VERSION_LIB,
    default_colombia_profile,
    linkage,
    load_source,
)
from record_linkage.exporters.smart import SmartExporter
from record_linkage.pipeline.result import PipelineResult


def _norm(texto: str) -> str:
    """Normaliza un nombre de columna: sin tildes, mayusculas, sin espacios extremos."""
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).strip().upper()


def leer_base(
    ruta: str | Path,
    hoja: str | int = 0,
    fila_encabezado: int = 0,
    columnas_texto: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Lee CSV/XLSX validando existencia y devolviendo columnas normalizadas.

    Args:
        ruta: ruta local al archivo .csv, .xlsx o .xls.
        hoja: nombre o indice de hoja (solo Excel).
        fila_encabezado: fila (0-index) donde estan los encabezados.
        columnas_texto: columnas a forzar como texto (p. ej. NIT, para no
            perder ceros a la izquierda ni convertir a notacion cientifica).

    Returns:
        DataFrame con encabezados normalizados (sin tildes, en mayusculas).

    Raises:
        FileNotFoundError: si la ruta no existe (con la carpeta que si existe).
        ValueError: si la extension no es soportada.
    """
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:12]) or "(vacia)"
        raise FileNotFoundError(
            f"No existe: {p}\n  La carpeta mas cercana que SI existe es: {padre}\n"
            f"  Contiene: {vistos}\n  Corrija la ruta o el nombre del archivo (copielo exacto)."
        )
    dtype = {c: "string" for c in columnas_texto}
    ext = p.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(p, header=fila_encabezado, dtype=dtype, encoding="utf-8")
    elif ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(p, sheet_name=hoja, header=fila_encabezado, dtype=dtype)
    else:
        raise ValueError(f"Extension no soportada: '{ext}'. Use .csv o .xlsx")
    df.columns = [_norm(c) for c in df.columns]
    return df


def exigir_columnas(df: pd.DataFrame, requeridas: dict[str, str], contexto: str) -> pd.DataFrame:
    """Renombra columnas del usuario al contrato NIT/RAZON_SOCIAL/... y valida.

    Args:
        df: base leida.
        requeridas: {nombre_en_su_archivo: nombre_contrato}; los nombres del
            archivo se comparan ya normalizados (sin tildes, mayusculas).
        contexto: etiqueta para mensajes de error (p. ej. "BASE_A").

    Returns:
        DataFrame solo con las columnas del contrato, renombradas.
    """
    mapa = {_norm(k): v for k, v in requeridas.items()}
    faltan = [k for k in mapa if k not in df.columns]
    if faltan:
        raise KeyError(
            f"[{contexto}] No se hallaron las columnas {faltan}.\n"
            f"  Columnas disponibles: {list(df.columns)}\n"
            "  Corrija los nombres en la CONFIGURACION (se comparan sin tildes ni mayusculas)."
        )
    out = df[list(mapa)].rename(columns=mapa).copy()
    for c in out.columns:
        out[c] = out[c].astype("string").fillna("").str.strip()
    return out


# ── Exportacion (Excel hasta ~1M filas por hoja; CSV comprimido si excede) ──

LIMITE_EXCEL = 1_000_000


def exportar_resultados(
    tablas: dict[str, pd.DataFrame],
    carpeta: str | Path,
    nombre_base: str,
) -> list[Path]:
    """Exporta con formulas neutralizadas; XLSX multi-hoja o CSV.gz por tabla.

    Returns:
        Rutas de los archivos escritos.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    escritos: list[Path] = []
    if all(len(t) <= LIMITE_EXCEL for t in tablas.values()):
        ruta = carpeta / f"{nombre_base}.xlsx"
        resultado = PipelineResult(work_dir=carpeta, extra=tablas)
        escritos.append(resultado.to_excel(ruta, include=tuple(tablas)))
    else:
        exporter = SmartExporter({"output_directory": str(carpeta)})
        for hoja, t in tablas.items():
            ruta = exporter.export(t, f"{nombre_base}__{hoja}", format="csv.gz")
            escritos.append(Path(ruta))
    for r in escritos:
        print(f"  Archivo escrito: {r}")
    return escritos


@dataclass(frozen=True)
class EspecBase:
    """Una fuente de entrada y el mapeo de SUS columnas al contrato del motor."""

    nombre: str
    archivo: str
    col_nit: str
    col_nombre: str
    col_ciudad: str | None = None
    hoja: str | int = 0
    fila_encabezado: int = 0
    formato: str = "auto"
    compresion: str = "auto"
    miembro_zip: str | None = None
    delimitador: str | None = None
    encoding: str = "auto"

    def __post_init__(self) -> None:
        if not self.nombre.strip():
            raise ValueError("EspecBase.nombre no puede estar vacio")

    def source_spec(self) -> SourceSpec:
        """Contrato seguro: proyecta columnas y descomprime ZIP a /content."""
        return SourceSpec(
            name=self.nombre,
            path=self.archivo,
            column_mapping={"NIT": self.col_nit, "RAZON_SOCIAL": self.col_nombre},
            optional_column_mapping={"CIUDAD": self.col_ciudad} if self.col_ciudad else {},
            column_types={"NIT": ColumnType.IDENTIFIER},
            format=self.formato,
            compression=self.compresion,
            archive_member=self.miembro_zip,
            delimiter=self.delimitador,
            encoding=self.encoding,
            sheet_name=self.hoja,
            header=self.fila_encabezado,
            chunksize=100_000,
            safety_limits=ZipSafetyLimits(
                max_members=16,
                max_member_bytes=1024**3,
                max_total_uncompressed_bytes=1024**3,
                max_compression_ratio=100.0,
            ),
            temp_dir="/content/rues_linker_tmp",
        )

    def cargar(self) -> LoadedSource:
        loaded = load_source(self.source_spec())
        df, reporte = loaded.data, loaded.report
        mb = df.memory_usage(deep=True).sum() / 1e6
        print(
            f"[{self.nombre}] {len(df):,} registros (~{mb:.0f} MB) | "
            f"{reporte.input_format.value}/{reporte.compression.value} | "
            f"encoding={reporte.encoding or '-'}"
        )
        for aviso in reporte.warnings:
            print(f"  Aviso de ingesta: {aviso}")
        return loaded


@dataclass
class ConfigProduccion:
    """Configuracion completa de la corrida multifuente (validada al construir).

    Attributes:
        etiqueta: identifica la corrida (carpeta de salida y manifiesto).
        fuentes: 2 o mas EspecBase.
        confiables: nombres de fuentes ya depuradas (subset de fuentes).
        perfil: perfil del Orchestrator (produccion_estandar / produccion_exhaustiva
            / alta_precision).
        usar_ciudad: activa CIUDAD si TODAS las fuentes la declaran.
        refinador_colombia: activa el MatchingProfile calibrado para Colombia
            (post-proceso que separa falsos positivos) + auditoria de decisiones.
        skip_reporting: omite L6; el notebook conserva sus exportaciones propias.
        collapse_exact_duplicates: procesa una copia por fila exacta y restaura la correlativa.
        dir_salida: entregables locales; dir_trabajo: checkpoints locales efimeros.
    """

    etiqueta: str
    fuentes: list[EspecBase]
    confiables: set[str] = field(default_factory=set)
    perfil: str = "produccion_estandar"
    usar_ciudad: bool = True
    refinador_colombia: bool = False
    skip_reporting: bool = True
    collapse_exact_duplicates: bool = True
    dir_salida: str = "/content/resultados/03"
    dir_trabajo: str = "/content/rues_linker_work"

    def __post_init__(self) -> None:
        if len(self.fuentes) < 2:
            raise ValueError("Se requieren al menos 2 fuentes (para 1 use el notebook 01)")
        nombres = [f.nombre for f in self.fuentes]
        if len(set(nombres)) != len(nombres):
            raise ValueError(f"Nombres de fuente repetidos: {nombres}")
        sobran = self.confiables - set(nombres)
        if sobran:
            raise ValueError(f"'confiables' contiene fuentes no declaradas: {sobran}")
        if not self.etiqueta.strip():
            raise ValueError("etiqueta no puede estar vacia")

    @property
    def carpeta_corrida(self) -> Path:
        return Path(self.dir_salida) / self.etiqueta

    @property
    def carpeta_trabajo(self) -> Path:
        return Path(self.dir_trabajo) / self.etiqueta


def _huella(df: pd.DataFrame, bloque: int = 100_000) -> str:
    """Huella completa y determinista, calculada por bloques para limitar RAM."""
    h = hashlib.sha256()
    h.update(f"{df.shape}|{list(df.columns)}".encode())
    for inicio in range(0, len(df), bloque):
        fin = min(inicio + bloque, len(df))
        hashes = pd.util.hash_pandas_object(df.iloc[inicio:fin], index=False)
        h.update(hashes.values.tobytes())
    return h.hexdigest()[:16]


def ejecutar_produccion(cfg: ConfigProduccion) -> dict[str, pd.DataFrame]:
    """Corrida completa: cargar N fuentes -> linkage() -> matriz -> manifiesto -> export."""
    t0 = time.time()
    cargadas = {f.nombre: f.cargar() for f in cfg.fuentes}
    fuentes = {nombre: loaded.data for nombre, loaded in cargadas.items()}
    con_ciudad = cfg.usar_ciudad and all("CIUDAD" in d.columns for d in fuentes.values())
    if cfg.usar_ciudad and not con_ciudad:
        print("Aviso: no todas las fuentes declaran ciudad; se corre SIN esa variable.")

    res = linkage(
        cargadas,
        trusted_sources=cfg.confiables or None,
        col_ciudad="CIUDAD" if con_ciudad else None,
        profile=cfg.perfil,
        matching_profile=default_colombia_profile() if cfg.refinador_colombia else None,
        return_matcher_audit=cfg.refinador_colombia,
        work_dir=str(cfg.carpeta_trabajo),
        skip_reporting=cfg.skip_reporting,
        collapse_exact_duplicates=cfg.collapse_exact_duplicates,
    )

    golden, correlativa = res["golden"], res["correlative"]
    presencia = (
        correlativa.assign(_uno=1)
        .pivot_table(index="ID_GRUPO", columns="SRC", values="_uno", aggfunc="max", fill_value=0)
        .reset_index()
    )
    resumen = pd.DataFrame(
        {
            "indicador": (
                [
                    "registros_totales",
                    "grupos_unicos",
                    "perfil",
                    "confiables",
                    "ciudad_usada",
                    "refinador_colombia",
                    "skip_reporting",
                    "collapse_exact_duplicates",
                    "duracion_seg",
                ]
                + [f"registros_{n}" for n in fuentes]
            ),
            "valor": (
                [
                    len(correlativa),
                    correlativa["ID_GRUPO"].nunique(),
                    cfg.perfil,
                    ",".join(sorted(cfg.confiables)) or "-",
                    con_ciudad,
                    cfg.refinador_colombia,
                    cfg.skip_reporting,
                    cfg.collapse_exact_duplicates,
                    round(time.time() - t0, 1),
                ]
                + [len(d) for d in fuentes.values()]
            ),
        }
    )
    tablas = {
        "GOLDEN": golden,
        "CORRELATIVA": correlativa,
        "MATRIZ_FUENTES": presencia,
        "RESUMEN": resumen,
    }
    if cfg.refinador_colombia and "matcher_decisions" in res:
        tablas["AUDITORIA_REFINADOR"] = res["matcher_decisions"]

    manifiesto = {
        "corrida": cfg.etiqueta,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "version_rues_linker": _VERSION_LIB,
        "perfil": cfg.perfil,
        "confiables": sorted(cfg.confiables),
        "ciudad_usada": con_ciudad,
        "refinador_colombia": cfg.refinador_colombia,
        "skip_reporting": cfg.skip_reporting,
        "collapse_exact_duplicates": cfg.collapse_exact_duplicates,
        "fuentes": {n: {"filas": len(d), "huella": _huella(d)} for n, d in fuentes.items()},
        "preprocesamiento": res.get("preprocessing", {}),
        "resultados": {
            "grupos": int(correlativa["ID_GRUPO"].nunique()),
            "registros": len(correlativa),
        },
    }
    cfg.carpeta_corrida.mkdir(parents=True, exist_ok=True)
    ruta_man = cfg.carpeta_corrida / "manifiesto_corrida.json"
    ruta_man.write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  Manifiesto: {ruta_man}")

    exportar_resultados(tablas, cfg.carpeta_corrida, f"consolidado_{cfg.etiqueta}")
    print(
        f"\nListo: {manifiesto['resultados']['grupos']:,} empresas unicas "
        f"a partir de {manifiesto['resultados']['registros']:,} registros "
        f"en {time.time() - t0:,.0f} s"
    )
    del cargadas, fuentes
    gc.collect()
    return tablas

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURAR Y EJECUTAR        (LA UNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
RAIZ = "/content/datos"

CONFIG = ConfigProduccion(
    etiqueta="CONSOLIDADO_2026_07",
    fuentes=[
        # ZIP con un CSV interno: declare el miembro para evitar ambiguedad.
        EspecBase(
            "RUES",
            f"{RAIZ}/rues.zip",
            col_nit="NIT",
            col_nombre="RAZON SOCIAL",
            col_ciudad="CIUDAD",
            compresion="zip",
            miembro_zip="rues.csv",
        ),
        # CSV: encoding='auto' valida UTF-8 y luego CP1252, siempre en modo estricto.
        EspecBase(
            "CRM", f"{RAIZ}/crm.csv", col_nit="NIT", col_nombre="NOMBRE", col_ciudad="CIUDAD"
        ),
        # TXT tabulado con encabezados propios.
        EspecBase(
            "EVENTOS",
            f"{RAIZ}/eventos.txt",
            col_nit="IDENTIFICACION",
            col_nombre="EMPRESA",
            formato="txt",
            delimitador="\t",
        ),
        EspecBase("GAZELLE", f"{RAIZ}/gazelle.xlsx", col_nit="NIT9", col_nombre="COMPANY"),
    ],
    confiables={"RUES"},
    perfil="produccion_estandar",  # produccion_exhaustiva | alta_precision
    refinador_colombia=False,  # True: post-proceso calibrado + AUDITORIA
    skip_reporting=True,  # ahorra RAM; las tablas de arriba SI se exportan
    collapse_exact_duplicates=True,  # acelera repetidos exactos sin perder filas de salida
)

tablas = ejecutar_produccion(CONFIG)
tablas["RESUMEN"]

### Operacion y reanudacion

- **Checkpoints**: cada fase queda en `/content/rues_linker_work/<etiqueta>/` y se
  reutiliza mientras el runtime siga vivo. Colab Free borra `/content` al terminar la sesion.
- **Corrida nueva sobre datos nuevos**: cambie la `etiqueta` (o borre su carpeta local).
- **MATRIZ_FUENTES**: por cada empresa unica, en que fuentes aparece (1/0) — ideal
  para responder "¿cuantas del CRM estan en RUES?" con una tabla dinamica.
- **Refinador Colombia**: si observa grupos "pegados" que no deberian (nombres genericos),
  active `refinador_colombia=True` y revise la hoja `AUDITORIA_REFINADOR` con la decision
  par a par.
- **Memoria**: `SourceSpec` proyecta solo las columnas declaradas y lee por bloques, pero
  `linkage()` materializa las fuentes para enlazarlas. Si no caben en RAM de Colab Free,
  reduzca/particione antes con `iter_source_chunks` o use un runtime con mas memoria.
- **Persistencia**: descargue `/content/resultados/03` antes de cerrar la sesion.